# Phylogenie XY

### 0. Create the conda env

In [ ]:
conda create -n phylo_sexlinked -c bioconda -c conda-forge \
blast mafft trimal iqtree seqkit -y

conda activate phylo_sexlinked

## 1.1 :  XY Phylogeny L.nobilis | L.azorica | L.novocanariensis commun sexed linked genes (>0.8)

In [ ]:
#!/bin/bash

############################
# 1. DIRECTORIES
############################

DB_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Blast_prot_db"
OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis"
FAA_DIR="$OUT_DIR/faa_prot"
BLAST_DIR="$OUT_DIR/blast"
PERSEA_BLAST_DIR="$BLAST_DIR/persea"
CINNA_BLAST_DIR="$BLAST_DIR/cinna"

mkdir -p "$DB_DIR" "$FAA_DIR" "$PERSEA_BLAST_DIR" "$CINNA_BLAST_DIR"

############################
# 2. INPUT FILES
############################

# Proteins
PERSEA_PROT="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/protein.faa"
CINNA_PROT="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/protein.faa"
# CDS
PERSEA_CDS="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/cds_from_genomic.fna"
CINNA_CDS="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/cds_from_genomic.fna"
# Query proteins
PROTEIN_FILE="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/braker.aa"
# Gametologs files 
WXYZ_nobilis="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"
WXYZ_novocanariensis="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out"
WXYZ_azorica="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_azorica_Lazorica_norm_filtered_EXONS.recode.out"
#Annotation file
ANNOTATION="/Users/cibio/Desktop/Laurus/Annotation/Annotation_final/braker.fixed_ids.gtf"

############################
# 1. STRAND FILE (PERMANENT)
############################

STRAND_FILE="$OUT_DIR/gene_strand.txt"

awk '
$3=="gene" {
    print $9 "\t" $7
}' "$ANNOTATION" > "$STRAND_FILE"

############################
# 3. BLAST DATABASES
############################

makeblastdb -in "$PERSEA_PROT" -dbtype prot -out "$DB_DIR/Persea_db"
makeblastdb -in "$CINNA_PROT" -dbtype prot -out "$DB_DIR/Cinna_db"

############################
# 4. GENES LIST
############################

genes=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)

############################
# 5. EXTRACTION + BLAST
############################

for g in "${genes[@]}"
do
    echo "Processing ${g}.t1"

    QUERY="$FAA_DIR/${g}.t1.faa"
    # Extraction
    awk -v gene=">${g}.t1" '
    $0 == gene {print; flag=1; next}
    /^>/ {flag=0}
    flag==1 {print}
    ' "$PROTEIN_FILE" > "$QUERY"

    # BLAST Persea
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Persea_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$PERSEA_BLAST_DIR/${g}_Persea.tsv"

    # BLAST Cinnamomum
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Cinna_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$CINNA_BLAST_DIR/${g}_Cinna.tsv"

done


############################
# 6. CDS + BEST HIT DIRECT PAR GENE
############################

for g in "${genes[@]}"
do
    echo "CDS extraction $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    mkdir -p "$GENE_DIR"

    #################################
    # BEST HITS (≥95% IDENTITY)
    #################################

    best_persea=$(awk '$3 >= 90' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | sort -k8,8nr | head -1 | cut -f2)
    best_cinna=$(awk '$3 >= 90' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | sort -k8,8nr | head -1 | cut -f2)

    echo -e "$g  PERSEA:$best_persea  CINNA:$best_cinna"

    #################################
    # SKIP IF NO OUTGROUP AT ALL
    #################################

    if [[ -z "$best_persea" && -z "$best_cinna" ]]; then
        echo "SKIP $g : no outgroup hit ≥95%"
        continue
    fi

    #################################
    # PERSEA CDS (if exists)
    #################################

    if [[ -n "$best_persea" ]]; then
        awk -v id="$best_persea" -v name="${g}_Persea_americana" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$PERSEA_CDS" > "$GENE_DIR/${g}_persea_CDS.fasta"
    else
        echo "WARNING $g : no Persea hit ≥95%"
    fi

    #################################
    # CINNA CDS (if exists)
    #################################

    if [[ -n "$best_cinna" ]]; then
        awk -v id="$best_cinna" -v name="${g}_Cinnamomum_micranthum" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$CINNA_CDS" > "$GENE_DIR/${g}_cinna_CDS.fasta"
    else
        echo "WARNING $g : no Cinna hit ≥95%"
    fi

done

############################
# 7. GAMETOLOGS X&Y PAR GENE
############################

for g in "${genes[@]}"
do
    echo "Processing $g"
    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    strand=$(grep -w "^$g" "$STRAND_FILE" | cut -f2)

    #########################
    # NOBILIS
    #########################

    grep -A 1 "${g}_X" "$WXYZ_nobilis" | \
    awk -v name="${g}_nobilis_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_nobilis_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_nobilis" | \
    awk -v name="${g}_nobilis_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_nobilis_Y.fasta"

    #########################
    # NOVOCANARIENSIS
    #########################

    grep -A 1 "${g}_X" "$WXYZ_novocanariensis" | \
    awk -v name="${g}_novocanariensis_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_novocanariensis_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_novocanariensis" | \
    awk -v name="${g}_novocanariensis_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_novocanariensis_Y.fasta"

    #########################
    # AZORICA
    #########################

    grep -A 1 "${g}_X" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_Y.fasta"

done


##########################################
# AUTO-DETECTION genes_1_1
# (≥1 outgroup with pident ≥95)
##########################################

genes_1_1=()
for g in "${genes[@]}"
do
    persea_hit=$(awk '$3 >= 90' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | head -1)
    cinna_hit=$(awk '$3 >= 90' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | head -1)
    if [[ -n "$persea_hit" || -n "$cinna_hit" ]]; then
        genes_1_1+=("$g")
    fi
done

echo "Genes retained for phylogeny:"
printf '%s\n' "${genes_1_1[@]}"

################################################
# 8. Concat gene + MACSE trimnonHomologous
################################################


# Gene avec orthologues dans au moins 1 outgroups
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)


for g in "${genes_1_1[@]}"
do
    echo "MACSE $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    #################################
    # CONCAT + FORMAT + MACSE
    #################################

    cat $GENE_DIR/*.fasta | \
    awk '
    /^>/ {
        if (seq) print seq
        print
        seq=""
        next
    }
    {
        seq = seq $0
    }
    END {
        if (seq) print seq
    }
    ' | \
    grep -v '^$' > $GENE_DIR/${g}_all.fasta


    #################################
    # MACSE
    #################################

    macse \
    -prog trimNonHomologousFragments \
    -seq $GENE_DIR/${g}_all.fasta \
    -out_NT $GENE_DIR/${g}_MACSE_clean.fasta \
    -min_trim_in 60 \
    -min_trim_ext 30

done

##########################################
# 9. MACSE alignSequences
##########################################

# Gene avec orthologues 1:1 + trmming avec 2 outgroups restants
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)

for g in "${genes_1_1[@]}"
do
    echo "ALIGNMENT $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    macse \
    -prog alignSequences \
    -seq "$GENE_DIR/${g}_MACSE_clean.fasta" \
    -out_NT "$GENE_DIR/${g}_aligned_NT.fasta" \
    -out_AA "$GENE_DIR/${g}_aligned_AA.fasta" \
    -max_refine_iter 10 

done

###########################################
# 10. GBLOCKS trimming
###########################################

for g in "${genes_1_1[@]}"
do
    echo "GBLOCKS $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    Gblocks "$GENE_DIR/${g}_aligned_NT.fasta" \
    -t=c 

done

###########################################
# 11. IQTREE 2 BEST MODEL 
###########################################


for g in "${genes_1_1[@]}"
do
    echo "TREE $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    ALIGN="$GENE_DIR/${g}_aligned_NT.fasta-gb.fa"

    ########################
    # 1. MODEL SELECTION
    ########################

    iqtree3 -s "$ALIGN" -m MFP -B 1000 -T AUTO

    #MODEL=$(grep "Best-fit model" "$ALIGN.iqtree" | awk '{print $5}')

done

In [ ]:
#######
    ########################
    # 2. RAXML-NG TREE
    ########################

sed 's/!/N/g' /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_aligned_NT.fasta-gb.fa > tmp && mv tmp /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_aligned_NT.fasta-gb.fa

raxml-ng \
    --all \
    --msa "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_aligned_NT.fasta-gb.fa" \
    --model GTR+G+FO \
    --prefix "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_RAxML" \
    --bs-trees autoMRE{2000} \
    --threads auto

raxml-ng \
    --all \
    --msa "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16717/g16717_aligned_NT.fasta-gb.fa" \
    --model GTR+G+FO \
    --prefix "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16717/g16717_RAxML" \
    --bs-trees autoMRE{2000} \
    --threads auto

### Stats BLAST

In [ ]:
##########################################
# INPUT DIR
##########################################

OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis"
PERSEA_BLAST_DIR="$OUT_DIR/blast/persea"
CINNA_BLAST_DIR="$OUT_DIR/blast/cinna"
STATS_OUT="$OUT_DIR/final_blast_stats.tsv"

##########################################
# HEADER
##########################################

echo -e "gene\tPersea_pident\tPersea_alnlen\tPersea_qlen\tPersea_slen\tPersea_bitscore\tCinna_pident\tCinna_alnlen\tCinna_qlen\tCinna_slen\tCinna_bitscore" > "$STATS_OUT"

##########################################
# LOOP OVER GENES (CLEAN VERSION)
##########################################

for GENE_DIR in "$OUT_DIR"/Gene_familly/*
do
    [[ -d "$GENE_DIR" ]] || continue
    g=$(basename "$GENE_DIR")

    #################################
    # BEST BLAST PERSEA
    #################################

    persea_line=$(awk '$3 >= 0' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | sort -k8,8nr | head -1)

    persea_pident=$(echo "$persea_line" | awk '{print $3}')
    persea_alnlen=$(echo "$persea_line" | awk '{print $4}')
    persea_qlen=$(echo "$persea_line" | awk '{print $5}')
    persea_slen=$(echo "$persea_line" | awk '{print $6}')
    persea_bitscore=$(echo "$persea_line" | awk '{print $8}')

    #################################
    # BEST BLAST CINNA
    #################################

    cinna_line=$(awk '$3 >= 0' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | sort -k8,8nr | head -1)

    cinna_pident=$(echo "$cinna_line" | awk '{print $3}')
    cinna_alnlen=$(echo "$cinna_line" | awk '{print $4}')
    cinna_qlen=$(echo "$cinna_line" | awk '{print $5}')
    cinna_slen=$(echo "$cinna_line" | awk '{print $6}')
    cinna_bitscore=$(echo "$cinna_line" | awk '{print $8}')

    echo -e "${g}\t${persea_pident}\t${persea_alnlen}\t${persea_qlen}\t${persea_slen}\t${persea_bitscore}\t${cinna_pident}\t${cinna_alnlen}\t${cinna_qlen}\t${cinna_slen}\t${cinna_bitscore}" >> "$STATS_OUT"
done

## 1.2 :  XY Phylogeny L.azorica | L.novocanariensis commun sexed linked genes (>0.8) (L_nobilis < 0.8)

In [ ]:
 #print(common_genes_azo_novo)
#g16665 g16678 g16681 g16684 g16687 g16729 g16731 g16732 g16736 g16747 g16750 g16774 g16806


genes=(g16665 g16678 g16681 g16684 g16687 g16729 g16731 g16732 g16736 g16747 g16750 g16774 g16806)
#genes=(g16732)
#genes=(g16681 g16684 g16729 g16747)
#genes_1_1=(g16681 g16684 g16729 g16747)


### script blast + phylogenie

In [ ]:
#!/bin/bash

############################
# 1. DIRECTORIES
############################

DB_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Blast_prot_db"
OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_azorica_novocanariensis"
FAA_DIR="$OUT_DIR/faa_prot"
BLAST_DIR="$OUT_DIR/blast"
PERSEA_BLAST_DIR="$BLAST_DIR/persea"
CINNA_BLAST_DIR="$BLAST_DIR/cinna"

mkdir -p "$DB_DIR" "$FAA_DIR" "$PERSEA_BLAST_DIR" "$CINNA_BLAST_DIR"

############################
# 2. INPUT FILES
############################

# Proteins
PERSEA_PROT="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/protein.faa"
CINNA_PROT="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/protein.faa"
# CDS
PERSEA_CDS="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/cds_from_genomic.fna"
CINNA_CDS="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/cds_from_genomic.fna"
# Query proteins
PROTEIN_FILE="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/braker.aa"
# Gametologs files 
WXYZ_novocanariensis="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out"
WXYZ_azorica="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_azorica_Lazorica_norm_filtered_EXONS.recode.out"
#Annotation file
ANNOTATION="/Users/cibio/Desktop/Laurus/Annotation/Annotation_final/braker.fixed_ids.gtf"

############################
# 1. STRAND FILE (PERMANENT)
############################

STRAND_FILE="$OUT_DIR/gene_strand.txt"

awk '
$3=="gene" {
    print $9 "\t" $7
}' "$ANNOTATION" > "$STRAND_FILE"

############################
# 3. BLAST DATABASES
############################

makeblastdb -in "$PERSEA_PROT" -dbtype prot -out "$DB_DIR/Persea_db"
makeblastdb -in "$CINNA_PROT" -dbtype prot -out "$DB_DIR/Cinna_db"

############################
# 4. GENES LIST
############################

genes=(g16665 g16678 g16681 g16684 g16687 g16729 g16731 g16732 g16736 g16747 g16750 g16774 g16806)

############################
# 5. EXTRACTION + BLAST
############################

for g in "${genes[@]}"
do
    echo "Processing ${g}.t1"

    QUERY="$FAA_DIR/${g}.t1.faa"
    # Extraction
    awk -v gene=">${g}.t1" '
    $0 == gene {print; flag=1; next}
    /^>/ {flag=0}
    flag==1 {print}
    ' "$PROTEIN_FILE" > "$QUERY"

    # BLAST Persea
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Persea_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$PERSEA_BLAST_DIR/${g}_Persea.tsv"

    # BLAST Cinnamomum
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Cinna_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$CINNA_BLAST_DIR/${g}_Cinna.tsv"

done


############################
# 6. CDS + BEST HIT DIRECT PAR GENE
############################

for g in "${genes[@]}"
do
    echo "CDS extraction $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    mkdir -p "$GENE_DIR"

    #################################
    # BEST HITS (≥90% IDENTITY)
    #################################

    best_persea=$(awk '$3 >= 90' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | sort -k8,8nr | head -1 | cut -f2)
    best_cinna=$(awk '$3 >= 90' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | sort -k8,8nr | head -1 | cut -f2)

    echo -e "$g  PERSEA:$best_persea  CINNA:$best_cinna"

    #################################
    # SKIP IF NO OUTGROUP AT ALL
    #################################

    if [[ -z "$best_persea" && -z "$best_cinna" ]]; then
        echo "SKIP $g : no outgroup hit ≥95%"
        continue
    fi

    #################################
    # PERSEA CDS (if exists)
    #################################

    if [[ -n "$best_persea" ]]; then
        awk -v id="$best_persea" -v name="${g}_Persea_americana" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$PERSEA_CDS" > "$GENE_DIR/${g}_persea_CDS.fasta"
    else
        echo "WARNING $g : no Persea hit ≥95%"
    fi

    #################################
    # CINNA CDS (if exists)
    #################################

    if [[ -n "$best_cinna" ]]; then
        awk -v id="$best_cinna" -v name="${g}_Cinnamomum_micranthum" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$CINNA_CDS" > "$GENE_DIR/${g}_cinna_CDS.fasta"
    else
        echo "WARNING $g : no Cinna hit ≥95%"
    fi

done

############################
# 7. GAMETOLOGS X&Y PAR GENE
############################

for g in "${genes[@]}"
do
    echo "Processing $g"
    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    strand=$(grep -w "^$g" "$STRAND_FILE" | cut -f2)

    #########################
    # NOVOCANARIENSIS
    #########################

    grep -A 1 "${g}_X" "$WXYZ_novocanariensis" | \
    awk -v name="${g}_novocanariensis_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_novocanariensis_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_novocanariensis" | \
    awk -v name="${g}_novocanariensis_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_novocanariensis_Y.fasta"

    #########################
    # AZORICA
    #########################

    grep -A 1 "${g}_X" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_Y.fasta"

done


##########################################
# AUTO-DETECTION genes_1_1
# (≥1 outgroup with pident ≥95)
##########################################

genes_1_1=()
for g in "${genes[@]}"
do
    persea_hit=$(awk '$3 >= 90' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | head -1)
    cinna_hit=$(awk '$3 >= 90' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | head -1)
    if [[ -n "$persea_hit" || -n "$cinna_hit" ]]; then
        genes_1_1+=("$g")
    fi
done

echo "Genes retained for phylogeny:"
printf '%s\n' "${genes_1_1[@]}"

################################################
# 8. Concat gene + MACSE trimnonHomologous
################################################


# Gene avec orthologues dans au moins 1 outgroups
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)


for g in "${genes_1_1[@]}"
do
    echo "MACSE $g"
    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    #################################
    # CONCAT + FORMAT + MACSE
    #################################

    cat $GENE_DIR/*.fasta | \
    awk '
    /^>/ {
        if (seq) print seq
        print
        seq=""
        next
    }
    {
        seq = seq $0
    }
    END {
        if (seq) print seq
    }
    ' | \
    grep -v '^$' > $GENE_DIR/${g}_all.fasta

    #################################
    # MACSE
    #################################

    macse \
    -prog trimNonHomologousFragments \
    -seq $GENE_DIR/${g}_all.fasta \
    -out_NT $GENE_DIR/${g}_MACSE_clean.fasta \
    -min_trim_in 60 \
    -min_trim_ext 30

done

##########################################
# 9. MACSE alignSequences
##########################################

# Gene avec orthologues 1:1 + trmming avec 2 outgroups restants
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)

for g in "${genes_1_1[@]}"
do
    echo "ALIGNMENT $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    macse \
    -prog alignSequences \
    -seq "$GENE_DIR/${g}_MACSE_clean.fasta" \
    -out_NT "$GENE_DIR/${g}_aligned_NT.fasta" \
    -out_AA "$GENE_DIR/${g}_aligned_AA.fasta" \
    -max_refine_iter 10 

done

###########################################
# 10. GBLOCKS trimming
###########################################

for g in "${genes_1_1[@]}"
do
    echo "GBLOCKS $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    Gblocks "$GENE_DIR/${g}_aligned_NT.fasta" \
    -t=c 

done

###########################################
# 11. IQTREE 2 BEST MODEL 
###########################################


for g in "${genes_1_1[@]}"
do
    echo "TREE $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    ALIGN="$GENE_DIR/${g}_aligned_NT.fasta-gb.fa"

    ########################
    # 1. MODEL SELECTION
    ########################

    iqtree3 -s "$ALIGN" -m MFP -B 1000 -T AUTO

    #MODEL=$(grep "Best-fit model" "$ALIGN.iqtree" | awk '{print $5}')

done

### Stats BLAST

In [ ]:
##########################################
# INPUT DIR
##########################################

OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_azorica_novocanariensis"
PERSEA_BLAST_DIR="$OUT_DIR/blast/persea"
CINNA_BLAST_DIR="$OUT_DIR/blast/cinna"
STATS_OUT="$OUT_DIR/final_phylo_stats.tsv"

##########################################
# HEADER
##########################################

echo -e "gene\tPersea_pident\tPersea_alnlen\tPersea_qlen\tPersea_slen\tPersea_bitscore\tCinna_pident\tCinna_alnlen\tCinna_qlen\tCinna_slen\tCinna_bitscore" > "$STATS_OUT"

##########################################
# LOOP OVER GENES (CLEAN VERSION)
##########################################

for GENE_DIR in "$OUT_DIR"/Gene_familly/*
do
    [[ -d "$GENE_DIR" ]] || continue
    g=$(basename "$GENE_DIR")

    #################################
    # BEST BLAST PERSEA
    #################################

    persea_line=$(awk '$3 >= 90' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | sort -k8,8nr | head -1)

    persea_pident=$(echo "$persea_line" | awk '{print $3}')
    persea_alnlen=$(echo "$persea_line" | awk '{print $4}')
    persea_qlen=$(echo "$persea_line" | awk '{print $5}')
    persea_slen=$(echo "$persea_line" | awk '{print $6}')
    persea_bitscore=$(echo "$persea_line" | awk '{print $8}')

    #################################
    # BEST BLAST CINNA
    #################################

    cinna_line=$(awk '$3 >= 90' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | sort -k8,8nr | head -1)

    cinna_pident=$(echo "$cinna_line" | awk '{print $3}')
    cinna_alnlen=$(echo "$cinna_line" | awk '{print $4}')
    cinna_qlen=$(echo "$cinna_line" | awk '{print $5}')
    cinna_slen=$(echo "$cinna_line" | awk '{print $6}')
    cinna_bitscore=$(echo "$cinna_line" | awk '{print $8}')

    echo -e "${g}\t${persea_pident}\t${persea_alnlen}\t${persea_qlen}\t${persea_slen}\t${persea_bitscore}\t${cinna_pident}\t${cinna_alnlen}\t${cinna_qlen}\t${cinna_slen}\t${cinna_bitscore}" >> "$STATS_OUT"
done

## 1.3 : XY Phylogeny L.azorica | L.nobilis commun sexed linked genes (>0.8) (L_novocanariensis < 0.8)

In [ ]:
genes=(g16692)
genes_1_1=(g16692)

### script blast + phylogenie

In [ ]:
#!/bin/bash

############################
# 1. DIRECTORIES
############################

DB_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Blast_prot_db"
OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_azorica_nobilis"
FAA_DIR="$OUT_DIR/faa_prot"
BLAST_DIR="$OUT_DIR/blast"
PERSEA_BLAST_DIR="$BLAST_DIR/persea"
CINNA_BLAST_DIR="$BLAST_DIR/cinna"

mkdir -p "$DB_DIR" "$FAA_DIR" "$PERSEA_BLAST_DIR" "$CINNA_BLAST_DIR"

############################
# 2. INPUT FILES
############################

# Proteins
PERSEA_PROT="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/protein.faa"
CINNA_PROT="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/protein.faa"
# CDS
PERSEA_CDS="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/cds_from_genomic.fna"
CINNA_CDS="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/cds_from_genomic.fna"
# Query proteins
PROTEIN_FILE="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/braker.aa"
# Gametologs files 
WXYZ_nobilis="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"
WXYZ_azorica="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_azorica_Lazorica_norm_filtered_EXONS.recode.out"
#Annotation file
ANNOTATION="/Users/cibio/Desktop/Laurus/Annotation/Annotation_final/braker.fixed_ids.gtf"

############################
# 1. STRAND FILE (PERMANENT)
############################

STRAND_FILE="$OUT_DIR/gene_strand.txt"

awk '
$3=="gene" {
    print $9 "\t" $7
}' "$ANNOTATION" > "$STRAND_FILE"

############################
# 3. BLAST DATABASES
############################

makeblastdb -in "$PERSEA_PROT" -dbtype prot -out "$DB_DIR/Persea_db"
makeblastdb -in "$CINNA_PROT" -dbtype prot -out "$DB_DIR/Cinna_db"

############################
# 4. GENES LIST
############################

genes=(g16692)

############################
# 5. EXTRACTION + BLAST
############################

for g in "${genes[@]}"
do
    echo "Processing ${g}.t1"

    QUERY="$FAA_DIR/${g}.t1.faa"
    # Extraction
    awk -v gene=">${g}.t1" '
    $0 == gene {print; flag=1; next}
    /^>/ {flag=0}
    flag==1 {print}
    ' "$PROTEIN_FILE" > "$QUERY"

    # BLAST Persea
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Persea_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$PERSEA_BLAST_DIR/${g}_Persea.tsv"

    # BLAST Cinnamomum
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Cinna_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$CINNA_BLAST_DIR/${g}_Cinna.tsv"

done


############################
# 6. CDS + BEST HIT DIRECT PAR GENE
############################

for g in "${genes[@]}"
do
    echo "CDS extraction $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    mkdir -p "$GENE_DIR"

    #################################
    # BEST HITS (≥90% IDENTITY)
    #################################

    best_persea=$(awk '$3 >= 90' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | sort -k8,8nr | head -1 | cut -f2)
    best_cinna=$(awk '$3 >= 90' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | sort -k8,8nr | head -1 | cut -f2)

    echo -e "$g  PERSEA:$best_persea  CINNA:$best_cinna"

    #################################
    # SKIP IF NO OUTGROUP AT ALL
    #################################

    if [[ -z "$best_persea" && -z "$best_cinna" ]]; then
        echo "SKIP $g : no outgroup hit ≥95%"
        continue
    fi

    #################################
    # PERSEA CDS (if exists)
    #################################

    if [[ -n "$best_persea" ]]; then
        awk -v id="$best_persea" -v name="${g}_Persea_americana" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$PERSEA_CDS" > "$GENE_DIR/${g}_persea_CDS.fasta"
    else
        echo "WARNING $g : no Persea hit ≥95%"
    fi

    #################################
    # CINNA CDS (if exists)
    #################################

    if [[ -n "$best_cinna" ]]; then
        awk -v id="$best_cinna" -v name="${g}_Cinnamomum_micranthum" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$CINNA_CDS" > "$GENE_DIR/${g}_cinna_CDS.fasta"
    else
        echo "WARNING $g : no Cinna hit ≥95%"
    fi

done

############################
# 7. GAMETOLOGS X&Y PAR GENE
############################

for g in "${genes[@]}"
do
    echo "Processing $g"
    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    strand=$(grep -w "^$g" "$STRAND_FILE" | cut -f2)


    #########################
    # NOBILIS
    #########################

    grep -A 1 "${g}_X" "$WXYZ_nobilis" | \
    awk -v name="${g}_nobilis_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_nobilis_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_nobilis" | \
    awk -v name="${g}_nobilis_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_nobilis_Y.fasta"


    #########################
    # AZORICA
    #########################

    grep -A 1 "${g}_X" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_Y.fasta"

done


##########################################
# AUTO-DETECTION genes_1_1
# (≥1 outgroup with pident ≥95)
##########################################

genes_1_1=()
for g in "${genes[@]}"
do
    persea_hit=$(awk '$3 >= 90' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | head -1)
    cinna_hit=$(awk '$3 >= 90' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | head -1)
    if [[ -n "$persea_hit" || -n "$cinna_hit" ]]; then
        genes_1_1+=("$g")
    fi
done

echo "Genes retained for phylogeny:"
printf '%s\n' "${genes_1_1[@]}"

################################################
# 8. Concat gene + MACSE trimnonHomologous
################################################


# Gene avec orthologues dans au moins 1 outgroups
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)


for g in "${genes_1_1[@]}"
do
    echo "MACSE $g"
    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    #################################
    # CONCAT + FORMAT + MACSE
    #################################

    cat $GENE_DIR/*.fasta | \
    awk '
    /^>/ {
        if (seq) print seq
        print
        seq=""
        next
    }
    {
        seq = seq $0
    }
    END {
        if (seq) print seq
    }
    ' | \
    grep -v '^$' > $GENE_DIR/${g}_all.fasta

    #################################
    # MACSE
    #################################

    macse \
    -prog trimNonHomologousFragments \
    -seq $GENE_DIR/${g}_all.fasta \
    -out_NT $GENE_DIR/${g}_MACSE_clean.fasta \
    -min_trim_in 60 \
    -min_trim_ext 30

done

##########################################
# 9. MACSE alignSequences
##########################################

# Gene avec orthologues 1:1 + trmming avec 2 outgroups restants
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)

for g in "${genes_1_1[@]}"
do
    echo "ALIGNMENT $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    macse \
    -prog alignSequences \
    -seq "$GENE_DIR/${g}_MACSE_clean.fasta" \
    -out_NT "$GENE_DIR/${g}_aligned_NT.fasta" \
    -out_AA "$GENE_DIR/${g}_aligned_AA.fasta" \
    -max_refine_iter 10 

done

###########################################
# 10. GBLOCKS trimming
###########################################

for g in "${genes_1_1[@]}"
do
    echo "GBLOCKS $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    Gblocks "$GENE_DIR/${g}_aligned_NT.fasta" \
    -t=c 

done

###########################################
# 11. IQTREE 2 BEST MODEL 
###########################################


for g in "${genes_1_1[@]}"
do
    echo "TREE $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    ALIGN="$GENE_DIR/${g}_aligned_NT.fasta-gb.fa"

    ########################
    # 1. MODEL SELECTION
    ########################

    iqtree3 -s "$ALIGN" -m MFP -B 1000 -T AUTO

    #MODEL=$(grep "Best-fit model" "$ALIGN.iqtree" | awk '{print $5}')

done

## 2 Phylogenie methode bayesienne

### Conversion gblock_fasta to phylip

In [ ]:
#!/bin/bash

INPUT=$1
OUTPUT=$2

awk '
BEGIN{seq="";taxa=0;sites=0}
/^>/{
    if(seq!=""){
        print seq
        if(length(seq)>sites) sites=length(seq)
        seq=""
    }
    taxa++
    # SUPPRESSION DU >
    sub(/^>/,"",$0)
    print
    next
}
{
    gsub(/[ \t]/,"")
    seq=seq$0
}

END{
    if(seq!="") print seq
    print taxa, sites > "header.tmp"
}
' "$INPUT" > seqs.tmp && \

sed 's/!/N/g' seqs.tmp > seqs_clean.tmp && \
cat header.tmp seqs_clean.tmp > "$OUTPUT" && \
rm -f header.tmp seqs.tmp seqs_clean.tmp

### Script pour lancer deux chaines et checker la convergence

In [ ]:
#!/bin/bash

ALIGN=$1
OUTDIR=$2
PREFIX="chain"
MINCYCLES=500
CHECKEVERY=100

mkdir -p "$OUTDIR"
cd "$OUTDIR"

echo "Starting chains on: $ALIGN"
echo "Output dir: $OUTDIR"

mpirun -np 2 pb_mpi -d "$ALIGN" -cat -gtr ${PREFIX}_1 > ${PREFIX}_1.log 2>&1 &
PID1=$!

mpirun -np 2 pb_mpi -d "$ALIGN" -cat -gtr ${PREFIX}_2 > ${PREFIX}_2.log 2>&1 &
PID2=$!

echo "PIDs : $PID1 $PID2"

CONVERGED=0

while [ $CONVERGED -eq 0 ]
do
    sleep 60
    if [ ! -f ${PREFIX}_1.trace ]; then
        echo "waiting trace file..."
        continue
    fi

    CYCLES=$(tail -n 1 ${PREFIX}_1.trace | awk '{print $1}')
    echo "cycles = ${CYCLES}"

    if [ "$CYCLES" -lt "$MINCYCLES" ]; then
        continue
    fi

    BURNIN=$((CYCLES / 5))

    echo "checking convergence"

    bpcomp -x ${BURNIN} 5 -o bpcomp ${PREFIX}_1 ${PREFIX}_2 > bpcomp.log 2>&1
    tracecomp -x ${BURNIN} ${PREFIX}_1 ${PREFIX}_2 > tracecomp.log 2>&1

    MAXDIFF=$(grep maxdiff bpcomp.bpdiff | awk '{print $3}')
    ESS=$(awk 'NR>2 {if(min=="" || $2 < min) min=$2} END {print min}' tracecomp.contdiff)

    echo "maxdiff = ${MAXDIFF}"
    echo "ESS = ${ESS}"

    OK1=$(awk -v x="$MAXDIFF" 'BEGIN{print (x<=0.1)?1:0}')
    OK2=$(awk -v x="$ESS" 'BEGIN{print (x>50)?1:0}')

    if [ "$OK1" -eq 1 ] && [ "$OK2" -eq 1 ]; then
        echo "Chains converged"

        kill $PID1
        kill $PID2

        #readpb -x ${BURNIN} 10 ${PREFIX}_1 ${PREFIX}_2

        CONVERGED=1
    fi
done

echo "Done"

### SCRIPT Phylobayes L.nobilis | L.azorica | L.novocanariensis commun sexed linked genes (>0.8)

In [ ]:
#!/bin/bash

OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis"
GENES=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16804 g16813 g16814)

for g in "${GENES[@]}"
do
    echo "========================"
    echo "Processing gene: $g"
    echo "========================"

    ############################
    # 1. DIRECTORIES
    ############################

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    ALIGN="$GENE_DIR/${g}_aligned_NT.fasta-gb.fa"

    PHYLO_DIR="$GENE_DIR/PhyloBayes"

    mkdir -p "$PHYLO_DIR"

    ############################
    # 2. CONVERSION FASTA -> PHYLOBAYES
    ############################

    echo "Converting alignment..."

    /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis/script_phylogbayes/Conversion_gblock_phy.sh \
        "$ALIGN" \
        "$PHYLO_DIR/${g}.phy"

    ############################
    # 3. RUN PHYLOBAYES PIPELINE
    ############################

    #echo "Running PhyloBayes..."

    /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis/script_phylogbayes/phylobayes.sh \
        "$PHYLO_DIR/${g}.phy" \
        "$PHYLO_DIR"

    ############################
    # 4. FINISH
    ############################

    echo "Done: $g"

done

### SCRIPT Phylobayes L.azorica | L.novocanariensis commun sexed linked genes (>0.8)

In [ ]:
#!/bin/bash

OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_azorica_novocanariensis"
GENES=(g16665 g16678 g16681 g16684 g16687 g16729 g16731 g16732 g16736 g16747 g16750 g16774 g16806)

for g in "${GENES[@]}"
do
    echo "========================"
    echo "Processing gene: $g"
    echo "========================"

    ############################
    # 1. DIRECTORIES
    ############################

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    ALIGN="$GENE_DIR/${g}_aligned_NT.fasta-gb.fa"

    PHYLO_DIR="$GENE_DIR/PhyloBayes"

    mkdir -p "$PHYLO_DIR"

    ############################
    # 2. CONVERSION FASTA -> PHYLOBAYES
    ############################

    echo "Converting alignment..."

    /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis/script_phylogbayes/Conversion_gblock_phy.sh \
        "$ALIGN" \
        "$PHYLO_DIR/${g}.phy"

    ############################
    # 3. RUN PHYLOBAYES PIPELINE
    ############################

    #echo "Running PhyloBayes..."

    /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis/script_phylogbayes/phylobayes.sh \
        "$PHYLO_DIR/${g}.phy" \
        "$PHYLO_DIR"

    ############################
    # 4. FINISH
    ############################

    echo "Done: $g"

done

### SCRIPT Phylobayes L.azorica | L.nobilis commun sexed linked genes (>0.8)

In [ ]:
#!/bin/bash

OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_azorica_nobilis"
GENES=(g16692)

for g in "${GENES[@]}"
do
    echo "========================"
    echo "Processing gene: $g"
    echo "========================"

    ############################
    # 1. DIRECTORIES
    ############################

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    ALIGN="$GENE_DIR/${g}_aligned_NT.fasta-gb.fa"

    PHYLO_DIR="$GENE_DIR/PhyloBayes"

    mkdir -p "$PHYLO_DIR"

    ############################
    # 2. CONVERSION FASTA -> PHYLOBAYES
    ############################

    echo "Converting alignment..."

    /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis/script_phylogbayes/Conversion_gblock_phy.sh \
        "$ALIGN" \
        "$PHYLO_DIR/${g}.phy"

    ############################
    # 3. RUN PHYLOBAYES PIPELINE
    ############################

    #echo "Running PhyloBayes..."

    /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes_nobilis_azorica_novocanariensis/script_phylogbayes/phylobayes.sh \
        "$PHYLO_DIR/${g}.phy" \
        "$PHYLO_DIR"

    ############################
    # 4. FINISH
    ############################

    echo "Done: $g"

done

## 3. BLAST GENE OUTLIER L.nobilis (g16607)

In [ ]:
#!/bin/bash

############################
# 1. DIRECTORIES
############################

DB_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Outlier_g16607/Blast_db"

mkdir -p "$DB_DIR"

############################
# 2. INPUT FILES
############################

GENOME_NOBILIS="/Users/cibio/Desktop/Laurus/genome/L_nobilis/ncbi_dataset/ncbi_dataset/data/GCA_048127095.1/GCA_048127095.1_ASM4812709v1_genomic.fna"
GENOME_AZORICA="/Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fasta"

############################
# 3. CREATE NUCLEOTIDE BLAST DATABASES
############################

echo "Creating BLAST database for Laurus nobilis..."

makeblastdb \
    -in "$GENOME_NOBILIS" \
    -dbtype nucl \
    -out "$DB_DIR/L_nobilis_db" \
    -parse_seqids

echo "Laurus nobilis database created."

echo "Creating BLAST database for Laurus azorica..."

makeblastdb \
    -in "$GENOME_AZORICA" \
    -dbtype nucl \
    -out "$DB_DIR/L_azorica_db" \
    -parse_seqids

echo "Laurus azorica database created."

echo "All nucleotide BLAST databases successfully created."

#################
#Blast g16607
#################

/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Outlier_g16607/g16607_X.fasta
/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Outlier_g16607/g16607_Y.fasta

# w

In [ ]:
#!/bin/bash

############################
# 1. DIRECTORIES
############################

BASE_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Outlier_g16607"

DB_DIR="$BASE_DIR/Blast_db"
BLAST_DIR="$BASE_DIR/Blast_results"

mkdir -p "$DB_DIR"
mkdir -p "$BLAST_DIR"

############################
# 2. INPUT FILES
############################

GENOME_NOBILIS="/Users/cibio/Desktop/Laurus/genome/L_nobilis/ncbi_dataset/ncbi_dataset/data/GCA_048127095.1/GCA_048127095.1_ASM4812709v1_genomic.fna"

GENOME_AZORICA="/Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fasta"

# Query sequences
G16607_X="$BASE_DIR/g16607_X.fasta"
G16607_Y="$BASE_DIR/g16607_Y.fasta"

############################
# 3. CREATE NUCLEOTIDE BLAST DATABASES
############################

echo "Creating BLAST database for Laurus nobilis..."

makeblastdb \
    -in "$GENOME_NOBILIS" \
    -dbtype nucl \
    -out "$DB_DIR/L_nobilis_db" \
    -parse_seqids

echo "Laurus nobilis database created."

echo "Creating BLAST database for Laurus azorica..."

makeblastdb \
    -in "$GENOME_AZORICA" \
    -dbtype nucl \
    -out "$DB_DIR/L_azorica_db" \
    -parse_seqids

echo "Laurus azorica database created."

############################
# 4. BLAST g16607 SEQUENCES
############################

echo "Running BLAST for g16607_X against L. nobilis..."

blastn \
    -query "$G16607_X" \
    -db "$DB_DIR/L_nobilis_db" \
    -out "$BLAST_DIR/g16607_X_vs_nobilis.tsv" \
    -evalue 1e-2 \
    -max_target_seqs 10 \
    -outfmt 6 \
    -num_threads 1

echo "Running BLAST for g16607_X against L. azorica..."

blastn \
    -query "$G16607_X" \
    -db "$DB_DIR/L_azorica_db" \
    -out "$BLAST_DIR/g16607_X_vs_azorica.tsv" \
    -outfmt 6 \
    -evalue 1e-2 \
    -max_target_seqs 10 \
    -num_threads 1

echo "Running BLAST for g16607_Y against L. nobilis..."

blastn \
    -query "$G16607_Y" \
    -db "$DB_DIR/L_nobilis_db" \
    -out "$BLAST_DIR/g16607_Y_vs_nobilis.tsv" \
    -outfmt 6 \
    -evalue 1e-2 \
    -max_target_seqs 10 \
    -num_threads 1

echo "Running BLAST for g16607_Y against L. azorica..."

blastn \
    -query "$G16607_Y" \
    -db "$DB_DIR/L_azorica_db" \
    -out "$BLAST_DIR/g16607_Y_vs_azorica.tsv" \
    -outfmt 6 \
    -evalue 1e-2 \
    -max_target_seqs 10 \
    -num_threads 1

echo "All BLAST searches completed."